# Notebook Huấn luyện & Đánh giá Mô hình Hồi quy (Regression Notebook)

Notebook này thực hiện huấn luyện, tinh chỉnh và đánh giá so sánh hiệu năng các thuật toán học máy hồi quy dự báo **số phút trễ thực tế** của chuyến bay trên dữ liệu chuẩn hóa của dự án Aeolus:
1. **Bài toán 1 (Cốt lõi)**: **Hồi quy Đến trễ (`ARR_DELAY` — số phút đến trễ)**
2. **Bài toán 2 (Phụ trợ)**: **Hồi quy Khởi hành trễ (`DEP_DELAY` — số phút khởi hành trễ)**

### 🤖 Các thuật toán được đánh giá ngang hàng:
* **XGBoost Regressor**: Mô hình Gradient Boosting mạnh mẽ (hỗ trợ GPU CUDA tự động).
* **Ridge Regression**: Mô hình tuyến tính L2-regularized làm baseline tham chiếu.
* **HistGradientBoostingRegressor**: Thay thế RandomForest, nhanh hơn 10x, hỗ trợ dữ liệu lớn.

### Cell 1: Import các thư viện cần thiết & Thiết lập môi trường

In [ ]:
import os
import sys
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Cấu hình cảnh báo và hiển thị
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', None)

# Hàm xác định thư mục gốc của repository
def resolve_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "data" / "split").exists() or (candidate / "src" / "data").exists():
            return candidate
    return Path.cwd()

repo_root = resolve_repo_root()
print(f"Repository Root: {repo_root}")
print("Tất cả các thư viện hồi quy đã được import thành công!")

### Cell 2: Hàm Nạp Dữ liệu Phân vùng Splits (Data Loader)
Hàm tự động nạp các tệp Parquet đã được chia sẵn theo mốc thời gian từ `preprocessing_notebook.ipynb`:
* **Train Set**: 2016 – 2022
* **Validation Set**: 2023
* **Test Set**: 2024 (Sealed Holdout)

In [ ]:
def load_regression_dataset(task_name):
    """
    Nạp dữ liệu từ thư mục split được tạo bởi preprocessing_notebook.
    Đọc X_*.parquet và y_*.parquet một cách ĐỘC LẬP (không đọc thư mục chung)
    để tránh lỗi schema mismatch giữa file X và file y.
    """
    candidate_base_dirs = [
        repo_root / "data" / "split" / task_name,
        repo_root / "src" / "data" / "split" / task_name
    ]
    
    task_dir = None
    for d in candidate_base_dirs:
        if d.exists():
            task_dir = d
            break
            
    if task_dir is None:
        raise FileNotFoundError(
            f"Không tìm thấy thư mục split '{task_name}'! "
            f"Vui lòng chạy 'preprocessing_notebook.ipynb' trước để tạo dữ liệu."
        )
        
    print(f"-> Đang nạp dữ liệu từ: {task_dir}")
    
    def load_fold(fold_name):
        fold_dir = task_dir / fold_name
        # Đọc riêng biệt X_*.parquet và y_*.parquet để tránh schema mismatch
        x_files = sorted(fold_dir.glob("X_*.parquet"))
        y_files = sorted(fold_dir.glob("y_*.parquet"))
        
        if not x_files or not y_files:
            raise FileNotFoundError(f"Không tìm thấy X_*.parquet hoặc y_*.parquet trong {fold_dir}")
        
        X = pd.concat([pd.read_parquet(f) for f in x_files], ignore_index=True)
        y = pd.concat([pd.read_parquet(f) for f in y_files], ignore_index=True)["target"]
        
        # Loại bỏ target column nếu vô tình có trong X
        drop_targets = [c for c in ["target", "IS_ARR_DELAY", "IS_DEP_DELAY", "ARR_DELAY", "DEP_DELAY"] if c in X.columns]
        if drop_targets:
            X = X.drop(columns=drop_targets)
        
        return X, y
    
    X_train, y_train = load_fold("train")
    X_valid, y_valid = load_fold("valid")
    X_test,  y_test  = load_fold("test")
        
    print(f"   * Kích thước Train (2016-2022) : X={X_train.shape}, y={y_train.shape} (Mean: {y_train.mean():.2f}m, Std: {y_train.std():.2f}m)")
    print(f"   * Kích thước Valid (2023)      : X={X_valid.shape}, y={y_valid.shape} (Mean: {y_valid.mean():.2f}m, Std: {y_valid.std():.2f}m)")
    print(f"   * Kích thước Test  (2024)      : X={X_test.shape}, y={y_test.shape} (Mean: {y_test.mean():.2f}m, Std: {y_test.std():.2f}m)")
    
    return X_train, y_train, X_valid, y_valid, X_test, y_test

print("Hàm load_regression_dataset đã sẵn sàng!")

### Cell 3: Hàm Huấn luyện & Đánh giá Mô hình Hồi quy (train_and_evaluate_regressor)

In [ ]:
all_regression_metrics = []

def train_and_evaluate_regressor(X_train, y_train, X_valid, y_valid, X_test, y_test, task_label, model_types=['xgb', 'ridge', 'rf']):
    print(f"\n{'='*70}")
    print(f"BẮT ĐẦU HUẤN LUYỆN BÀI TOÁN HỒI QUY: {task_label.upper()}")
    print(f"CÁC THUẬT TOÁN: {model_types}")
    print(f"{'='*70}")
    
    # Chuẩn hóa các biến số liên tục (fit một lần duy nhất trên train)
    scaler = StandardScaler()
    X_train_scaled = X_train.copy()
    X_valid_scaled = X_valid.copy()
    X_test_scaled  = X_test.copy()
    
    numeric_cols = [c for c in X_train.columns if X_train[c].dtype in ['float32', 'float64']]
    if numeric_cols:
        X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
        X_valid_scaled[numeric_cols] = scaler.transform(X_valid[numeric_cols])
        X_test_scaled[numeric_cols]  = scaler.transform(X_test[numeric_cols])
        
    trained_models = {}
    trained_scalers = {}
    
    for m_type in model_types:
        print(f"\n{'-'*50}")
        print(f"Đang cấu hình và huấn luyện mô hình: {m_type.upper()}...")
        
        if m_type == 'xgb':
            import xgboost as xgb
            use_device = 'cuda'
            try:
                test_reg = xgb.XGBRegressor(n_estimators=1, device='cuda')
                test_reg.fit(X_train_scaled.iloc[:5], y_train.iloc[:5])
                print("-> Kích hoạt thành công tăng tốc GPU (CUDA) cho XGBoost Regressor!")
            except Exception:
                print("-> GPU không khả dụng, chuyển sang sử dụng CPU đa luồng...")
                use_device = 'cpu'
                
            reg = xgb.XGBRegressor(
                n_estimators=150,
                max_depth=8,
                learning_rate=0.08,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                tree_method='hist',
                device=use_device,
                n_jobs=-1 if use_device == 'cpu' else None
            )
        elif m_type == 'ridge':
            reg = Ridge(alpha=1.0)
        elif m_type == 'rf':
            # Sử dụng HistGradientBoostingRegressor thay vì RandomForest thuần để tăng tốc độ gấp 10 lần
            reg = HistGradientBoostingRegressor(max_iter=100, max_depth=10, random_state=42)
        else:
            print(f"Bỏ qua: Mô hình '{m_type}' không hợp lệ.")
            continue
            
        reg.fit(X_train_scaled, y_train)
        
        # Đánh giá trên Validation (2023)
        y_val_pred = reg.predict(X_valid_scaled)
        val_mae = mean_absolute_error(y_valid, y_val_pred)
        val_rmse = np.sqrt(mean_squared_error(y_valid, y_val_pred))
        val_r2 = r2_score(y_valid, y_val_pred)
        print(f"Validation 2023 -> MAE: {val_mae:.2f} phút | RMSE: {val_rmse:.2f} phút | R2: {val_r2:.4f}")
        
        # Đánh giá trên Test (2024)
        y_test_pred = reg.predict(X_test_scaled)
        test_mae = mean_absolute_error(y_test, y_test_pred)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
        test_r2 = r2_score(y_test, y_test_pred)
        
        # Tính tỷ lệ sai lệch trong phạm vi dung sai vận hành sân bay
        errors = np.abs(y_test - y_test_pred)
        within_15m = (errors <= 15).mean() * 100
        within_30m = (errors <= 30).mean() * 100
        
        print(f"Test 2024       -> MAE: {test_mae:.2f} phút | RMSE: {test_rmse:.2f} phút | R2: {test_r2:.4f}")
        print(f"Dung sai vận hành: {within_15m:.2f}% chuyến bay trong sai số ±15m | {within_30m:.2f}% trong sai số ±30m")
        
        all_regression_metrics.append({
            "Task": task_label,
            "Model": m_type.upper(),
            "Val MAE (m)": round(val_mae, 2),
            "Test MAE (m)": round(test_mae, 2),
            "Test RMSE (m)": round(test_rmse, 2),
            "Test R2": round(test_r2, 4),
            "Error <= 15m": f"{within_15m:.1f}%",
            "Error <= 30m": f"{within_30m:.1f}%"
        })
        
        trained_models[m_type] = reg
        trained_scalers[m_type] = scaler  # Lưu scaler cùng mô hình để dùng trong visualization
        
    return trained_models, trained_scalers

print("Hàm train_and_evaluate_regressor đã sẵn sàng!")

### Cell 4: Thực nghiệm Bài toán 1 (Cốt lõi) - Hồi quy Đến trễ (ARR_DELAY)

In [ ]:
# Nạp dữ liệu bài toán Hồi quy Đến trễ
X_tr_arr, y_tr_arr, X_va_arr, y_va_arr, X_te_arr, y_te_arr = load_regression_dataset("arrival_regression")

# Huấn luyện và đánh giá các mô hình hồi quy
reg_models_arrival, reg_scalers_arrival = train_and_evaluate_regressor(
    X_tr_arr, y_tr_arr,
    X_va_arr, y_va_arr,
    X_te_arr, y_te_arr,
    task_label="ARR_DELAY (phút)",
    model_types=['xgb', 'ridge', 'rf']
)

### Cell 5: Thực nghiệm Bài toán 2 (Phụ trợ) - Hồi quy Khởi hành trễ (DEP_DELAY)

In [ ]:
# Nạp dữ liệu bài toán Hồi quy Khởi hành trễ
X_tr_dep, y_tr_dep, X_va_dep, y_va_dep, X_te_dep, y_te_dep = load_regression_dataset("departure_regression")

# Huấn luyện và đánh giá các mô hình hồi quy
reg_models_departure, reg_scalers_departure = train_and_evaluate_regressor(
    X_tr_dep, y_tr_dep,
    X_va_dep, y_va_dep,
    X_te_dep, y_te_dep,
    task_label="DEP_DELAY (phút)",
    model_types=['xgb', 'ridge', 'rf']
)

### Cell 6: Bảng So sánh Tổng hợp Hiệu năng các Mô hình Hồi quy

In [ ]:
df_summary_reg = pd.DataFrame(all_regression_metrics)
print("=" * 95)
print("TỔNG HỢP SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH HỒI QUY (REGRESSION BENCHMARK)")
print("=" * 95)
print(df_summary_reg.to_string(index=False))

### Cell 7: Trực quan hóa Phân phối Sai số Dự báo (Residual Distribution)
Kiểm tra mức độ tập trung sai số quanh 0 và tỷ lệ dự báo nằm trong dung sai ±15 phút vận hành sân bay.

In [ ]:
if 'xgb' in reg_models_arrival:
    xgb_reg = reg_models_arrival['xgb']
    scaler_vis = reg_scalers_arrival['xgb']  # Dùng đúng scaler đã fit trong quá trình train
    
    numeric_cols = [c for c in X_tr_arr.columns if X_tr_arr[c].dtype in ['float32', 'float64']]
    X_te_scaled = X_te_arr.copy()
    if numeric_cols:
        X_te_scaled[numeric_cols] = scaler_vis.transform(X_te_arr[numeric_cols])
        
    y_pred_arr = xgb_reg.predict(X_te_scaled)
    residuals = (y_te_arr - y_pred_arr).clip(-60, 60)
    
    plt.figure(figsize=(10, 4))
    plt.hist(residuals, bins=60, color='mediumseagreen', edgecolor='black', alpha=0.7)
    plt.axvline(0, color='red', linestyle='--', linewidth=2, label="Sai số = 0")
    plt.axvline(15, color='orange', linestyle=':', label="Dung sai +15m")
    plt.axvline(-15, color='orange', linestyle=':', label="Dung sai -15m")
    plt.title("Phân phối Sai số Dự báo ARR_DELAY (Test 2024) - XGBoost")
    plt.xlabel("Sai số dự báo = Thực tế - Dự đoán (phút)")
    plt.ylabel("Số lượng chuyến bay")
    plt.legend()
    plt.tight_layout()
    plt.show()